# Off-the-shelf benchmarks: Logistic Regression, Random Forest, XGBoost

MT (`initial-mt.ipynb`) shares one representation across all five factors. This notebook asks
whether that sharing actually buys anything, by benchmarking against three standard classifiers
that estimate a completely separate model per factor, with no shared structure at all (paper
Sec 3.2.3). Same walk-forward procedure as the MT notebook — expanding training window, 2-year
validation window immediately before the test year, single held-out test year — so results are
directly comparable.

**What this notebook does:**
1. Load the same response factors / macro / financial predictor panel as `initial-mt.ipynb`,
   via the shared `est`/`loading` helpers.
2. **Logistic Regression** (paper's "LR"): plain, unregularized — the paper doesn't grid-search
   LR either, so none is done here.
3. **Random Forest**: fixed hyperparameters (500 trees, depth 5, sqrt(p) features per split)
   in place of the paper's per-fold grid search, for tractability.
4. **XGBoost** (stand-in for the paper's "GBT"): fixed hyperparameters (200 trees, depth 2,
   learning rate 0.1, 50% subsample), also fixed rather than grid-searched.
5. Benchmark OOS classification accuracy (paper Table 1) and multi-factor timing Sharpe ratio
   (paper Table 3) for all three against the paper's published numbers, and against this
   project's MT results if `results/mt_oos_predictions.csv` already exists.

**Not included**: the paper's single-task LSTM benchmark — retraining it from scratch for all
32 folds x 5 factors was too slow to be worth it here, so it's dropped rather than run to
completion (see the README's "Out of scope" section).

**Random Forest comes out of this notebook well ahead of the paper's own RF and MT numbers** —
see `rf-investigation.ipynb` for a dedicated diagnostic on why (short version: not seed luck,
not crisis-avoidance, not feature leakage — genuine skill on HML/SMB/CMA sourced mostly from
the financial/anomaly predictors, not the macro panel).

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier

import loading
import estimation as est

FACTOR_NAMES = est.FACTOR_NAMES
warnings.filterwarnings('ignore', category=ConvergenceWarning)  # unregularized LR on ~250-600 rows x 259 features warns often; expected, not a bug
warnings.filterwarnings('ignore', category=FutureWarning)  # sklearn's LogisticRegression(penalty=None) deprecation warns on every one of the 160 walk-forward fits

In [2]:
# Off-the-shelf models (paper Sec 3.2.3): unlike MT, each of these estimates a *separate*
# functional form per factor, with no shared structure across factors. Same walk-forward
# estimation procedure as the MT notebook (paper Sec 3.2.4) — expanding training window,
# 2-year validation window immediately before the test year, single held-out test year — so
# results are directly comparable to "initial-mt.ipynb".

data, feature_cols = est.build_labels_and_panel(
    loading.response_factors, loading.macro_predictors, loading.financial_predictors
)

print("data shape:", data.shape)
print("features:", len(feature_cols))
print("date range:", data.index.min().date(), "to", data.index.max().date())

data shape: (683, 264)
features: 259
date range: 1965-01-31 to 2021-11-30


In [3]:
# --- Logistic Regression (paper's "LR") ---
# Plain, unregularized LR — the paper's Table IA1 lists a hyperparameter grid for the
# penalized version (EN) but not for LR, so no tuning is performed here either.

def lr_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:  # degenerate fold guard: a factor with a one-sided sign run
        return np.full(len(X_test), y_train.mean())
    model = LogisticRegression(penalty=None, max_iter=5000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]

lr_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, lr_fit_predict)
    for f in FACTOR_NAMES
})
lr_predictions.to_csv('../results/lr_oos_predictions.csv')
print("LR done:", lr_predictions.shape)
lr_predictions.head()

LR done: (383, 5)


,SMB_prob,HML_prob,RMW_prob,CMA_prob,MOM_prob
1990-01-31,1.975874e-39,1.000000e+00,8.371336e-06,7.997366e-11,1.000000e+00
1990-02-28,1.436394e-25,1.000000e+00,8.174403e-30,1.000000e+00,9.288759e-01
1990-03-31,1.000000e+00,1.000000e+00,9.346918e-20,1.578330e-03,9.999954e-01
1990-04-30,7.748952e-03,1.631418e-65,1.000000e+00,6.498412e-52,2.325447e-32
1990-05-31,1.000000e+00,7.281014e-29,1.000000e+00,2.742651e-16,5.862422e-33


In [4]:
# --- Random Forest ---
# Fixed hyperparameters (500 trees, depth 5, sqrt(p) features per split) rather than the
# paper's per-fold grid search over Table IA1's {#Trees, Depth} grid, for tractability —
# same simplification as the MT notebook's fixed (l1, learning rate).

def rf_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:
        return np.full(len(X_test), y_train.mean())
    model = RandomForestClassifier(
        n_estimators=500, max_depth=5, max_features='sqrt', random_state=0, n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]

rf_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, rf_fit_predict)
    for f in FACTOR_NAMES
})
rf_predictions.to_csv('../results/rf_oos_predictions.csv')
print("RF done:", rf_predictions.shape)
rf_predictions.head()

RF done: (383, 5)


,SMB_prob,HML_prob,RMW_prob,CMA_prob,MOM_prob
1990-01-31,0.365743,0.623755,0.444696,0.623499,0.632201
1990-02-28,0.578191,0.660460,0.531975,0.583773,0.632404
1990-03-31,0.663023,0.516440,0.593175,0.469144,0.666596
1990-04-30,0.574674,0.530922,0.593777,0.527786,0.567019
1990-05-31,0.593805,0.468371,0.575374,0.496614,0.583377


In [5]:
# --- XGBoost (stand-in for the paper's "GBT") ---
# Fixed hyperparameters (200 trees, depth 2, learning rate 0.1, 50% subsample — all valid
# points in Table IA1's GBT grid) rather than a per-fold grid search, for tractability.

def xgb_fit_predict(X_train, y_train, X_val, y_val, X_test):
    if len(np.unique(y_train)) < 2:
        return np.full(len(X_test), y_train.mean())
    model = XGBClassifier(
        n_estimators=200, max_depth=2, learning_rate=0.1, subsample=0.5,
        eval_metric='logloss', random_state=0,
    )
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]

xgb_predictions = pd.DataFrame({
    f'{f}_prob': est.walk_forward_predict_single_task(data, feature_cols, f, xgb_fit_predict)
    for f in FACTOR_NAMES
})
xgb_predictions.to_csv('../results/xgb_oos_predictions.csv')
print("XGBoost done:", xgb_predictions.shape)
xgb_predictions.head()

XGBoost done: (383, 5)


,SMB_prob,HML_prob,RMW_prob,CMA_prob,MOM_prob
1990-01-31,0.523065,0.905987,0.166423,0.786694,0.910435
1990-02-28,0.506134,0.883377,0.605608,0.866616,0.617017
1990-03-31,0.822050,0.261250,0.712173,0.390512,0.851008
1990-04-30,0.666736,0.204737,0.394075,0.092950,0.456548
1990-05-31,0.349383,0.430814,0.749792,0.410605,0.232397


In [6]:
# Benchmark: OOS classification accuracy (paper Table 1) and multi-factor timing Sharpe ratio
# (paper Table 3) for each off-the-shelf model, alongside the paper's published numbers and
# (if it has been run) this project's MT model for reference.
#
# Note: the paper's single-task LSTM benchmark is not included here — retraining it from
# scratch for all 32 folds x 5 factors was too slow to be worth it for this pass, so it's
# dropped rather than run to completion.

models = {'LR': lr_predictions, 'RF': rf_predictions, 'XGBoost (GBT)': xgb_predictions}

try:
    mt_predictions = pd.read_csv('../results/mt_oos_predictions.csv', index_col=0, parse_dates=True)
    models['MT'] = mt_predictions
except FileNotFoundError:
    pass

rows = []
for name, pred in models.items():
    rows.append(est.benchmark_summary(pred, data, loading.response_factors, name))
summary = pd.DataFrame(rows).set_index('Model')

paper_reference = pd.DataFrame({
    'Mean Accuracy': {'LR': 53.1, 'RF': 55.6, 'XGBoost (GBT)': 54.8, 'MT': 55.4},
    'Sharpe Ratio': {'LR': 0.61, 'RF': 0.66, 'XGBoost (GBT)': 0.61, 'MT': 0.69},
}).T

print("This notebook, out-of-sample 1990-2021:")
print(summary.round(3))
print("\nPaper's published numbers, for reference (accuracy %, Sharpe ratio):")
print(paper_reference)

This notebook, out-of-sample 1990-2021:
               Mean Accuracy  Sharpe Ratio  alpha (annualized %)  t(alpha)  \
Model                                                                        
LR                     0.511         0.638                 0.877     1.532   
RF                     0.563         0.844                 1.344     3.636   
XGBoost (GBT)          0.546         0.792                 1.431     2.693   
MT                     0.543         0.666                 0.680     1.473   

                beta  R2 (%)  
Model                         
LR             0.436  40.836  
RF             0.767  79.360  
XGBoost (GBT)  0.502  46.252  
MT             0.669  68.991  

Paper's published numbers, for reference (accuracy %, Sharpe ratio):
                  LR     RF  XGBoost (GBT)     MT
Mean Accuracy  53.10  55.60          54.80  55.40
Sharpe Ratio    0.61   0.66           0.61   0.69
